In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from run_csl_daily_annotation import process_single_image
from mmpose.apis import init_model

In [ ]:
# Initialize RTMPose3D model
print('Initializing RTMPose3D model...')
pose_config = 'demo/mmpose/projects/rtmpose3d/configs/rtmw3d-l_8xb64_cocktail14-384x288.py'
pose_checkpoint = 'demo/mmpose/projects/rtmpose3d/demo/rtmw3d-l_8xb64_cocktail14-384x288-794dbc78_20240626.pth'
device = 'cuda:0'  # Change to 'cpu' if no GPU available

pose_estimator = init_model(pose_config, pose_checkpoint, device=device)
print('Model loaded successfully')

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from run_csl_daily_annotation import process_single_image
from mmpose.apis import init_model

def plot_hand_camera_beautiful(image, joints, camera_position, camera_intrinsic, boundary=False, draw_skeleton=True):
    """Beautiful hand keypoint visualization with enhanced colors and effects"""
    fx, fy, cx, cy = camera_intrinsic 
    
    # Calculate positions relative to camera
    joints_camera = joints - camera_position
    
    # Project to image plane
    joints_image = joints_camera.copy()
    joints_image[..., 0] = joints_image[..., 0] / joints_image[..., 2] * fx + cx
    joints_image[..., 1] = joints_image[..., 1] / joints_image[..., 2] * fy + cy
    
    # Calculate depth values for coloring
    valid_joints = ~np.isnan(joints_camera[:, 2])
    joints_depth = np.zeros_like(joints_camera[:, 2])
    if valid_joints.any():
        joints_depth[valid_joints] = 1 - (joints_camera[valid_joints, 2] - joints_camera[valid_joints, 2].min()) / (joints_camera[valid_joints, 2].max() - joints_camera[valid_joints, 2].min())

    # Enhanced skeleton connection definitions with colors
    skeleton_connections = {
        'arm': [(0, 1), (1, 2)],  # 手臂连接
        'palm': [(2, 3), (3, 8), (8, 12), (12, 16), (16, 20), (20, 3)],  # 手掌连接
        'thumb': [(3, 4), (4, 5), (5, 6), (6, 7)],    # 拇指
        'index': [(8, 9), (9, 10), (10, 11)],         # 食指
        'middle': [(12, 13), (13, 14), (14, 15)],     # 中指
        'ring': [(16, 17), (17, 18), (18, 19)],       # 无名指
        'pinky': [(20, 21), (21, 22), (22, 23)]       # 小指
    }
    
    # Define beautiful colors for different parts
    colors = {
        'arm': (180, 60, 60),        # 深红色系 - 手臂
        'palm': (60, 180, 60),       # 深绿色系 - 手掌
        'thumb': (180, 120, 60),     # 深橙色系 - 拇指
        'index': (60, 90, 180),      # 深蓝色系 - 食指
        'middle': (180, 60, 180),    # 深紫色系 - 中指
        'ring': (60, 180, 180),      # 深青色系 - 无名指
        'pinky': (180, 180, 60)      # 深黄色系 - 小指
    }
    # Joint type colors
    joint_colors = {
        'arm': (220, 80, 80),
        'palm': (80, 220, 80),
        'finger': (80, 120, 220)
    }

    items_to_draw = []
    
    # Draw boundary first (furthest back)
    if boundary:
        valid_joints = ~np.isnan(joints_camera).any(axis=1)
        if valid_joints.any():
            boundary = np.concatenate([
                joints_camera[valid_joints].min(0), 
                joints_camera[valid_joints].max(0)
            ], 0)
            boundary_line = get_boundary(boundary)
            boundary_line[..., 0] = boundary_line[..., 0] / boundary_line[..., 2] * fx + cx
            boundary_line[..., 1] = boundary_line[..., 1] / boundary_line[..., 2] * fy + cy
            for line in boundary_line:
                pt1 = line[0]
                pt2 = line[1]
                items_to_draw.append([
                    (pt1[:2].astype(np.int32), pt2[:2].astype(np.int32)), 
                    'boundary', 
                    max(pt1[2], pt2[2]),
                    (150, 150, 150),  # Gray color for boundary
                    1
                ])
    
    if draw_skeleton:
        # Draw skeleton connections with different colors
        for part_name, connections in skeleton_connections.items():
            color = colors[part_name]
            for i, j in connections:
                # Skip if either point is NaN
                if np.isnan(joints_image[i]).any() or np.isnan(joints_image[j]).any():
                    continue
                pt1 = joints_image[i]
                pt2 = joints_image[j]
                
                # Calculate line thickness based on depth and part type
                thickness = 5 if part_name == 'arm' else 4
                
                items_to_draw.append([
                    (pt1[:2].astype(np.int32), pt2[:2].astype(np.int32)), 
                    'line', 
                    max(joints_camera[i, 2], joints_camera[j, 2]),
                    color,
                    thickness
                ])
    
    # Draw joints with enhanced styling
    for i in range(len(joints_image)):
        # Skip if point is NaN
        if np.isnan(joints_image[i]).any():
            continue
            
        pt = joints_image[i]
        depth_ratio = joints_depth[i] if i < len(joints_depth) else 0.5
        
        # Determine joint type and color
        if i < 3:  # Arm joints
            color = joint_colors['arm']
            radius = 8
        elif i == 3:  # Palm center
            color = joint_colors['palm']
            radius = 7
        else:  # Finger joints
            color = joint_colors['finger']
            radius = 6
        
        # Add depth-based brightness variation
        brightness = 0.7 + 0.3 * depth_ratio
        enhanced_color = tuple(int(c * brightness) for c in color)
        
        items_to_draw.append([
            (pt[:2].astype(np.int32), radius), 
            'point', 
            joints_camera[i, 2],
            enhanced_color,
            i  # joint index for special handling
        ])
    
    # Sort by z-depth for proper layering (larger z values drawn first - further back)
    items_to_draw.sort(key=lambda x: -x[2])
    
    # Draw items with enhanced effects
    for item in items_to_draw:
        if item[1] == 'line':
            pt1, pt2 = item[0]
            color = item[3]
            thickness = item[4]
            
            # # Draw shadow effect
            # shadow_offset = 2
            # shadow_color = (50, 50, 50)
            # cv2.line(image, 
            #         (pt1[0] + shadow_offset, pt1[1] + shadow_offset), 
            #         (pt2[0] + shadow_offset, pt2[1] + shadow_offset), 
            #         shadow_color, thickness, 
            #         lineType=cv2.LINE_AA, 
            #         shift=0)
            
            # Draw main line with dashed effect
            dash_length = 10
            dx = pt2[0] - pt1[0]
            dy = pt2[1] - pt1[1]
            dist = np.sqrt(dx*dx + dy*dy)
            
            if dist > 0:
                # Calculate number of dashes based on distance
                num_dashes = max(2, int(dist / (2 * dash_length)))
                
                # Draw dashed line segments
                for i in range(num_dashes):
                    start_ratio = i / num_dashes
                    end_ratio = (i + 0.5) / num_dashes  # Only draw half of each segment for dashed effect
                    
                    start_x = int(pt1[0] + dx * start_ratio)
                    start_y = int(pt1[1] + dy * start_ratio)
                    end_x = int(pt1[0] + dx * end_ratio)
                    end_y = int(pt1[1] + dy * end_ratio)
                    
                    cv2.line(image, 
                            (start_x, start_y), 
                            (end_x, end_y), 
                            color, thickness,
                            lineType=cv2.LINE_AA)
            
        elif item[1] == 'point':
            center, radius = item[0]
            color = item[3]
            joint_idx = item[4]
            
            # Draw shadow
            shadow_offset = 2
            shadow_color = (50, 50, 50)
            cv2.circle(image, 
                      (center[0] + shadow_offset, center[1] + shadow_offset), 
                      radius, shadow_color, -1)
            
            # Draw outer ring
            outer_color = tuple(max(0, c - 40) for c in color)
            cv2.circle(image, tuple(center), radius + 1, outer_color, 2)
            
            # Draw main circle
            cv2.circle(image, tuple(center), radius, color, -1)
            
            # Draw inner highlight
            highlight_color = tuple(min(255, c + 60) for c in color)
            cv2.circle(image, 
                      (center[0] - 1, center[1] - 1), 
                      max(1, radius - 2), highlight_color, -1)
            
        elif item[1] == 'boundary':
            pt1, pt2 = item[0]
            color = item[3]
            thickness = item[4]
            cv2.line(image, tuple(pt1), tuple(pt2), color, thickness)
    
    return image


def get_boundary(boundary):
    """Generate boundary box lines for visualization"""
    x0, y0, z0, x1, y1, z1 = boundary + np.array([-10, -10, -10, 10, 10, 10])
    boundary_line = np.array([
        [x0, y0, z0], [x0, y0, z1], 
        [x0, y1, z0], [x0, y1, z1], 
        [x1, y0, z0], [x1, y0, z1], 
        [x1, y1, z0], [x1, y1, z1], 
        [x0, y0, z0], [x1, y0, z0], 
        [x0, y1, z0], [x1, y1, z0], 
        [x0, y0, z1], [x1, y0, z1], 
        [x0, y1, z1], [x1, y1, z1], 
        [x0, y0, z0], [x0, y1, z0], 
        [x1, y0, z0], [x1, y1, z0], 
        [x0, y0, z1], [x0, y1, z1], 
        [x1, y0, z1], [x1, y1, z1]
    ]).reshape(-1, 2, 3)
    return boundary_line

In [ ]:
# Read the output.png image
input_img = cv2.imread('output.png')
if input_img is None:
    print('Error: Could not read output.png')
else:
    print(f'Successfully loaded image with shape: {input_img.shape}')

# Create a simple args object for the process_single_image function
class Args:
    def __init__(self):
        self.kpt_thr = 0.5
        self.device = device

args = Args()

# Process the image to get keypoints
print('Estimating poses...')
pose_results = process_single_image(input_img, pose_estimator, args)

# Extract keypoints and scores
keypoints = pose_results.pred_instances.keypoints[0]  # Shape: [133, 3] (x, y, z)
keypoint_scores = pose_results.pred_instances.keypoint_scores[0]  # Shape: [133]

print(f'Detected keypoints shape: {keypoints.shape}')
print(f'Keypoint scores shape: {keypoint_scores.shape}')

# # Set low confidence keypoints to NaN
# low_conf_mask = keypoint_scores < args.kpt_thr
# keypoints[low_conf_mask] = np.nan

# Process depth center (normalize z coordinates)
# depths_center = keypoints[[6, 7], 2].mean()
# keypoints[..., 2] = keypoints[..., 2] - depths_center
center = keypoints[[6, 7]].mean(axis=0)
# keypoints[..., 2] = keypoints[..., 2] - center[2]
keypoints = keypoints - center

# Extract body and hand keypoints
keypoints_processed = np.concatenate([
    keypoints[:17],    # body keypoints
    keypoints[-42:],   # hand keypoints (both hands)
], axis=0)  # Shape: [59, 3]

# Extract arm and hand joints for visualization
def process_pose_for_visualization(pose):
    """Extract arm and hand joints from pose data for visualization"""
    left_arm_hand = np.concatenate([pose[[5,7,9], :], pose[-42:-21, :]], axis=0)  # [24, 3]
    right_arm_hand = np.concatenate([pose[[6,8,10], :], pose[-21:, :]], axis=0)   # [24, 3]
    return left_arm_hand, right_arm_hand

left_joints, right_joints = process_pose_for_visualization(keypoints_processed)

print(f'Left hand joints shape: {left_joints.shape}')
print(f'Right hand joints shape: {right_joints.shape}')

# Create annotated image
annotated_img = input_img.copy()

# Camera parameters for projection (adjust these if needed)
camera_params = {
    'camera_position': np.array([-400, -500, -4000]),
    'camera_intrinsic': np.array([800, 800, input_img.shape[1]/2, input_img.shape[0]/2])
}

# Display using matplotlib
plt.figure(figsize=(5, 5), dpi=300)

# Create a clean visualization
clean_img = np.ones((input_img.shape[0], input_img.shape[1], 3), dtype=np.uint8) * 255
clean_img = plot_hand_camera_beautiful(
    clean_img, 
    left_joints * 1000,
    **camera_params,
    boundary=False,
    draw_skeleton=True
)
clean_img = plot_hand_camera_beautiful(
    clean_img, 
    right_joints * 1000,
    **camera_params,
    boundary=False,
    draw_skeleton=True
)
plt.imshow(cv2.cvtColor(clean_img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.tight_layout()
plt.savefig('keypoint_comparison.pdf', dpi=300)
plt.show()